In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import os
import json
import sys
import numpy as np
sys.path.append("..")
import copy

In [3]:
from src import models, data, operators, editors, functional, metrics, lens
from src.utils import logging_utils, experiment_utils
import logging
import torch
import baukit

logging_utils.configure(level=logging.INFO)

In [4]:
mt = models.load_model("gptj", fp16=True, device="cuda")

2026-04-25 16:52:15 src.models INFO     loading EleutherAI/gpt-j-6B (device=cuda, fp16=True)
2026-04-25 16:52:15 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6B/resolve/float16/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6b/resolve/float16/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/EleutherAI/gpt-j-6b/b71ae8bc86cac13154e03e92b5855203086b722e/config.json "HTTP/1.1 200 OK"


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6B/resolve/float16/model.safetensors "HTTP/1.1 307 Temporary Redirect"


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

[transformers] GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6B/resolve/float16/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6b/resolve/float16/generation_config.json "HTTP/1.1 404 Not Found"


2026-04-25 16:52:16 huggingface_hub.utils._http WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6B/resolve/float16/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6b/resolve/float16/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-25 16:52:16 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/EleutherAI/gpt-j-6b/b71ae8bc86cac13154e03e92b5855203086b722e/config.json "HTTP/1.1 200 OK"
2026-04-25 16:52:18 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-25 16:52:19 httpx INFO     HTTP Request: HEAD https://huggingface.co/EleutherAI/gpt-j-6b/resolve/main/config.json "HTTP/1.1 307 Te

In [5]:
dataset = data.load_dataset()

In [6]:
##################################
layer = 3
rank = 170
beta = 2.25
n_train = 5
selected_relations = [r for r in dataset if r.name in [
        "person occupation",
        "name birthplace",
        "person university"
    ]
]

experiment_utils.set_seed(123456)
##################################


relation_properties = {}

for relation in selected_relations:
    train, test = relation.split(n_train)
    prompt_template = relation.prompt_templates[0]

    relation_prompt = functional.make_prompt(
        mt=mt,
        prompt_template=prompt_template,
        subject="{}",
        examples=train.samples,
    )

    estimator = operators.JacobianIclMeanEstimator(
        mt = mt, h_layer=layer, beta=beta, rank=rank
    )
    operator = estimator(train)

    relation_properties[relation.name] = {
        "train": train,
        "prompt_template": prompt_template,
        "prompt": relation_prompt,
        "operator": operator,
    }

2026-04-25 16:52:22 src.utils.experiment_utils INFO     setting all seeds to 123456
2026-04-25 16:52:57 src.operators WARNING  relation has > 1 prompt_templates, will use first ({} works as a)
2026-04-25 16:53:25 src.operators WARNING  relation has > 1 prompt_templates, will use first (For university, {} attended)


In [7]:
for relation_name in relation_properties:
    print("-----------------------------------")
    print(relation_name)
    print("-----------------------------------")
    print(f"{[sample.__str__() for sample in relation_properties[relation_name]['train'].samples]}")
    print(relation_properties[relation_name]['prompt'])

-----------------------------------
name birthplace
-----------------------------------
['Rohit -> India', 'Sakura -> Japan', 'Marco -> Italy', 'Hong -> China', 'Kraipob -> Thailand']
<|endoftext|>Rohit was born in the country of India
Sakura was born in the country of Japan
Marco was born in the country of Italy
Hong was born in the country of China
Kraipob was born in the country of Thailand
{} was born in the country of
-----------------------------------
person occupation
-----------------------------------
['Yakubu Gowon -> politician', 'Geraldine McNulty -> actor', 'Andrew Salkey -> poet', 'Wilhelm Magnus -> mathematician', 'Samuel Medary -> journalist']
<|endoftext|>Yakubu Gowon works as a politician
Geraldine McNulty works as a actor
Andrew Salkey works as a poet
Wilhelm Magnus works as a mathematician
Samuel Medary works as a journalist
{} works as a
-----------------------------------
person university
-----------------------------------
['Ursula K. Le Guin -> Columbia Univer

In [8]:
for relation_name in relation_properties:
    relation_properties[relation_name]["W_inv"] = functional.low_rank_pinv(
        matrix = relation_properties[relation_name]["operator"].weight,
        rank=rank,
    )

In [9]:
##################################################
source_subject = "X"
targ_prop_for_subj = {
    "person occupation": "Sherlock Holmes",
    "name birthplace": "Jackie Chan",
    "person university": "Bill Gates",  
}
##################################################

for prop in targ_prop_for_subj:
    prompt = relation_properties[prop]['prompt'].format(source_subject)
    obj = functional.predict_next_token(mt = mt, prompt = prompt, k=3)[0]
    print(f"{source_subject} -- {prop} -- {obj[0].__str__()}")
print("=================================")

for prop, subj in targ_prop_for_subj.items():
    prompt = relation_properties[prop]['prompt'].format(subj)
    obj = functional.predict_next_token(mt = mt, prompt = prompt, k=3)[0]
    print(f"{subj} -- {prop} -- {obj[0].__str__()}")

X -- person occupation --  politician (p=0.066)
X -- name birthplace --  Russia (p=0.076)
X -- person university --  University (p=0.347)
Sherlock Holmes -- person occupation --  detective (p=0.523)
Jackie Chan -- name birthplace --  China (p=0.581)
Bill Gates -- person university --  Harvard (p=0.893)


In [10]:
def get_delta_s(
    prop, 
    source_subject, 
    target_subject,
    fix_latent_norm = None,
):
    w_p_inv = relation_properties[prop]["W_inv"]
    hs_and_zs = functional.compute_hs_and_zs(
        mt = mt,
        prompt_template = relation_properties[prop]["prompt_template"],
        subjects = [source_subject, target_subject],
        h_layer= layer,
        z_layer=-1,
        examples= relation_properties[prop]["train"].samples,
    )

    z_source = hs_and_zs.z_by_subj[source_subject]
    z_target = hs_and_zs.z_by_subj[targ_prop_for_subj[prop]]
    # print(z_target.norm().item(), z_source.norm().item())

    h_source = hs_and_zs.h_by_subj[source_subject]
    h_target = hs_and_zs.h_by_subj[targ_prop_for_subj[prop]]

    z_source *= fix_latent_norm / z_source.norm() if fix_latent_norm is not None else 1.0
    z_target *= z_source.norm() / z_target.norm() if fix_latent_norm is not None else 1.0
    print(z_target.norm().item(), z_source.norm().item())

    delta_s = w_p_inv @  (z_target.squeeze() - z_source.squeeze())
    
    print(f"h_source: {h_source.norm().item()} | h_target: {h_target.norm().item()}")
    print(f"inv_h_source: {(w_p_inv @ z_source).norm().item()} | inv_h_target: {(w_p_inv @ z_target).norm().item()}")
    print(delta_s.norm().item())

    return delta_s, hs_and_zs

In [11]:
delta_s_by_prop = {}
for relation_name in targ_prop_for_subj:
    delta_s, hs_and_zs = get_delta_s(
        prop = relation_name,
        source_subject = source_subject,
        target_subject = targ_prop_for_subj[relation_name],
        # fix_latent_norm=250
    )

    delta_s_by_prop[relation_name] = {
        "delta_s": delta_s,
        "hs_and_zs": hs_and_zs,
    }

313.0 363.75
h_source: 46.5 | h_target: 53.28125
inv_h_source: 75.0 | inv_h_target: 74.625
50.1875
241.125 319.0
h_source: 46.25 | h_target: 51.75
inv_h_source: 52.9375 | inv_h_target: 54.8125
46.0
159.25 350.25
h_source: 43.84375 | h_target: 50.03125
inv_h_source: 62.21875 | inv_h_target: 66.4375
75.6875


In [12]:
drr = [prop["delta_s"] for relation, prop in delta_s_by_prop.items()]
for i in range(len(drr)):
    for j in range(len(drr)):
        print(f"{i} -- {j} -- {(drr[i][None] @ drr[j][None].T).squeeze().item()}, {torch.cosine_similarity(drr[i], drr[j], dim=0).item()}")

0 -- 0 -- 2520.0, 1.0
0 -- 1 -- 142.0, 0.0615234375
0 -- 2 -- -17.875, -0.004711151123046875
1 -- 0 -- 142.0, 0.0615234375
1 -- 1 -- 2116.0, 1.0
1 -- 2 -- -56.4375, -0.016204833984375
2 -- 0 -- -17.875, -0.004711151123046875
2 -- 1 -- -56.4375, -0.016204833984375
2 -- 2 -- 5728.0, 0.99951171875


In [13]:
max_norm = np.array([delta_s_by_prop[relation_name]["delta_s"].norm().item() for relation_name in delta_s_by_prop.keys()]).max()

cumulative_delta_s = torch.zeros_like(delta_s_by_prop[prop]["delta_s"])
for relation_name in delta_s_by_prop:
    ds = delta_s_by_prop[relation_name]["delta_s"]
    ds = ds*max_norm / ds.norm()
    delta_s_by_prop[relation_name]["delta_s"] = ds
    cumulative_delta_s += ds
cumulative_delta_s /= 3
cumulative_delta_s.shape, cumulative_delta_s.norm().item()

(torch.Size([4096]), 44.28125)

In [14]:
# relation_names = [
#         "person occupation",
#         "name birthplace",
#         "person university"
# ]

# cumulative_delta_s = (
#     delta_s_by_prop[relation_names[0]]["delta_s"] + 
#     delta_s_by_prop[relation_names[1]]["delta_s"] + 
#     delta_s_by_prop[relation_names[2]]["delta_s"]
# )

In [15]:
# prop = "person occupation"
# prop = "name birthplace"
prop = "person university"

delta_s, hs_and_zs = get_delta_s(
    prop = prop,
    source_subject = source_subject,
    target_subject = targ_prop_for_subj[prop],
    fix_latent_norm = 250
)

def get_intervention(h, int_layer, subj_idx):
    def edit_output(output, layer):
        if(layer != int_layer):
            return output
        functional.untuple(output)[:, subj_idx] = h
        return output
    return edit_output

prompt = relation_properties[prop]["prompt"].format(source_subject)

h_index, inputs = functional.find_subject_token_index(
    mt=mt,
    prompt=prompt,
    subject=source_subject,
)

h_layer, z_layer = models.determine_layer_paths(model = mt, layers = [layer, -1])

with baukit.TraceDict(
    mt.model, layers = [h_layer, z_layer],
    edit_output=get_intervention(
        # h = hs_and_zs.h_by_subj[source_subject]
        # h = hs_and_zs.h_by_subj[source_subject] + delta_s,
        h = hs_and_zs.h_by_subj[source_subject] + cumulative_delta_s, 
        int_layer = h_layer, 
        subj_idx = h_index
    )
) as traces:
    outputs = mt.model(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
    )

lens.interpret_logits(
    mt = mt, 
    logits = outputs.logits[0][-1], 
    # get_proba=True
)

250.125 250.0
h_source: 43.84375 | h_target: 50.03125
inv_h_source: 44.40625 | inv_h_target: 104.375
99.9375


[(' Beijing', 17.516),
 (' University', 17.125),
 (' the', 16.906),
 (' P', 16.906),
 (' T', 16.562),
 (' St', 16.312),
 (' Harvard', 16.266),
 (' Hang', 15.906),
 (' Shanghai', 15.617),
 (' Sun', 15.578)]